# Fine-tune (LoRA)

Fine-tunes **Qwen2.5-0.5B-Instruct** on GSM8K using LoRA. Artifacts go to `output/fine_tune/` (gitignored).

| Output | Contents |
|--------|---------|
| Cell output | tqdm bar + `[train] step=… loss=… lr=… epoch=…` per log event |
| `training_run.log` | Timestamped log, appended each run |
| `trainer_log_history.json` | Per-step `loss`, `learning_rate`, `epoch` from HF Trainer |
| `training_summary.json` | Hyperparameters + final loss / step / epoch |
| `adapter_*/`, checkpoints | Saved by HF Trainer under the run folder |

**Single run:** `RUN_HYPERPARAMETER_SWEEP = False` — uses defaults from `hyperparameters.py`.  
**Sweep:** `True` — one run per entry in `PARAM_COMBINATIONS`, each in its own subfolder (`run_000_…`).

Run cells top to bottom.

### 1 — Imports and callbacks

Imports the stack and defines `NotebookProgressCallback` (prints `[train]` lines per log event) and `LossConvergenceCallback` (early stopping). Run once per session.

In [2]:
import json
import logging
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from transformers import TrainerCallback

from qwen_math_flow.download_model import download_qwen_25_07b
from qwen_math_flow.load_dataset import format_gsm8k_as_chat, load_and_tokenize_math
from qwen_math_flow.lora_finetune import LossConvergenceCallback, create_lora_model, run_finetune


class NotebookProgressCallback(TrainerCallback):
    """Print one line per Trainer log so loss / lr / epoch show in the notebook while training."""

    def on_train_begin(self, args, state, control, **kwargs):
        print(
            "[train] started | "
            f"epochs={args.num_train_epochs} | log every {args.logging_steps} step(s) | "
            "progress bar + metrics lines below",
            flush=True,
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        parts = [f"step={state.global_step}"]
        if logs.get("loss") is not None:
            parts.append(f"loss={logs['loss']:.4f}")
        if logs.get("learning_rate") is not None:
            parts.append(f"lr={logs['learning_rate']:.2e}")
        if logs.get("epoch") is not None:
            parts.append(f"epoch={logs['epoch']:.4f}")
        print("[train] " + " | ".join(parts), flush=True)

    def on_train_end(self, args, state, control, **kwargs):
        print(f"[train] finished | global_step={state.global_step}", flush=True)

### 2 — Config

Set the output path, sweep toggle, and logging verbosity. Training defaults (LR, LoRA rank, batch size, etc.) come from `hyperparameters.py`; entries in `PARAM_COMBINATIONS` override only the keys they list.

Re-run this cell whenever you change sweep settings.

In [3]:
from itertools import product

from qwen_math_flow.hyperparameters import (
    DATASET_NAME,
    DATASET_SPLIT,
    GRADIENT_ACCUMULATION_STEPS,
    LEARNING_RATE,
    LOAD_IN_4BIT,
    LORA_ALPHA,
    LORA_R,
    MAX_LENGTH,
    MAX_TRAIN_SAMPLES,
    MODEL_CACHE_DIR,
    NUM_EPOCHS,
    PER_DEVICE_TRAIN_BATCH_SIZE,
)

OUTPUT_BASE = "output/fine_tune"

# False: one run using hyperparameters.py only. True: one run per dict in PARAM_COMBINATIONS.
RUN_HYPERPARAMETER_SWEEP = True

# Sweeps: each dict can override training keys; missing keys use hyperparameters.py defaults.
PARAM_COMBINATIONS = [
    {"learning_rate": 1e-5, "lora_r": 8,  "lora_alpha": 16, "num_epochs": 1},
    {"learning_rate": 2e-5, "lora_r": 8,  "lora_alpha": 16, "num_epochs": 2},
    {"learning_rate": 2e-5, "lora_r": 16, "lora_alpha": 32, "num_epochs": 2},
    {"learning_rate": 3e-5, "lora_r": 16, "lora_alpha": 32, "num_epochs": 3},
    {"learning_rate": 3e-5, "lora_r": 32, "lora_alpha": 64, "num_epochs": 3},
]
# Example — Cartesian grid (2×2×2 = 8 combinations). Replace PARAM_COMBINATIONS above or assign to it.
# PARAM_COMBINATIONS = [
#     {"learning_rate": lr, "lora_r": r, "lora_alpha": r * 2, "num_epochs": ep}
#     for lr, r, ep in product([1e-5, 2e-5], [8, 16], [1, 2])
# ]


def merged_training_params(overrides: dict) -> dict:
    """Defaults from hyperparameters.py, overridden by one sweep row."""
    return {
        "learning_rate": overrides.get("learning_rate", LEARNING_RATE),
        "lora_r": overrides.get("lora_r", LORA_R),
        "lora_alpha": overrides.get("lora_alpha", LORA_ALPHA),
        "num_epochs": overrides.get("num_epochs", NUM_EPOCHS),
        "per_device_train_batch_size": overrides.get(
            "per_device_train_batch_size", PER_DEVICE_TRAIN_BATCH_SIZE
        ),
        "gradient_accumulation_steps": overrides.get(
            "gradient_accumulation_steps", GRADIENT_ACCUMULATION_STEPS
        ),
        "max_length": overrides.get("max_length", MAX_LENGTH),
        "max_train_samples": overrides.get("max_train_samples", MAX_TRAIN_SAMPLES),
    }


def combination_subdir(index: int, m: dict) -> str:
    """Unique folder name for one combination."""
    lr = m["learning_rate"]
    lr_s = f"{lr:.0e}".replace("e-0", "e-").replace("e+0", "e+")
    return f"run_{index:03d}_lr{lr_s}_r{m['lora_r']}_a{m['lora_alpha']}_ep{m['num_epochs']}"


SWEEP_RUNS = PARAM_COMBINATIONS if RUN_HYPERPARAMETER_SWEEP else [{}]
NUM_COMBINATIONS = len(SWEEP_RUNS)

# Trainer: log loss / lr every N steps (1 = every step — most verbose)
LOGGING_STEPS = 1
SHOW_NOTEBOOK_LOG_LINES = True
DISABLE_TQDM = False
TRANSFORMERS_DEBUG = False

Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)
print(f"Output base: {Path(OUTPUT_BASE).resolve()}")
print(f"RUN_HYPERPARAMETER_SWEEP={RUN_HYPERPARAMETER_SWEEP}  →  {NUM_COMBINATIONS} training run(s)")
if RUN_HYPERPARAMETER_SWEEP:
    for i, o in enumerate(PARAM_COMBINATIONS):
        print(f"  [{i}] {o}  →  {combination_subdir(i, merged_training_params(o))}")
print(
    f"LOGGING_STEPS={LOGGING_STEPS}  SHOW_NOTEBOOK_LOG_LINES={SHOW_NOTEBOOK_LOG_LINES}  "
    f"DISABLE_TQDM={DISABLE_TQDM}  TRANSFORMERS_DEBUG={TRANSFORMERS_DEBUG}"
)

Output base: /common/home/users/c/chinyee.lew.2022/output/fine_tune
RUN_HYPERPARAMETER_SWEEP=True  →  5 training run(s)
  [0] {'learning_rate': 1e-05, 'lora_r': 8, 'lora_alpha': 16, 'num_epochs': 1}  →  run_000_lr1e-5_r8_a16_ep1
  [1] {'learning_rate': 2e-05, 'lora_r': 8, 'lora_alpha': 16, 'num_epochs': 2}  →  run_001_lr2e-5_r8_a16_ep2
  [2] {'learning_rate': 2e-05, 'lora_r': 16, 'lora_alpha': 32, 'num_epochs': 2}  →  run_002_lr2e-5_r16_a32_ep2
  [3] {'learning_rate': 3e-05, 'lora_r': 16, 'lora_alpha': 32, 'num_epochs': 3}  →  run_003_lr3e-5_r16_a32_ep3
  [4] {'learning_rate': 3e-05, 'lora_r': 32, 'lora_alpha': 64, 'num_epochs': 3}  →  run_004_lr3e-5_r32_a64_ep3
LOGGING_STEPS=1  SHOW_NOTEBOOK_LOG_LINES=True  DISABLE_TQDM=False  TRANSFORMERS_DEBUG=False


### 3 — Train

Loads the model and data, runs `run_finetune` for each combination, and writes `training_summary.json`, `trainer_log_history.json`, and `training_run.log` per run. In sweep mode, also writes `sweep_index.json` at the top level.

Re-run config first if hyperparameters changed.

In [4]:
def _setup_run_logging(log_path: Path) -> None:
    logging.basicConfig(
        level=logging.DEBUG if TRANSFORMERS_DEBUG else logging.INFO,
        format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(log_path, encoding="utf-8", mode="a"),
        ],
        force=True,
    )
    for name in ("transformers", "transformers.trainer", "qwen_math_flow.lora_finetune"):
        logging.getLogger(name).setLevel(logging.DEBUG if TRANSFORMERS_DEBUG else logging.INFO)
    logging.getLogger("datasets").setLevel(logging.WARNING)
    if TRANSFORMERS_DEBUG:
        transformers.logging.set_verbosity_debug()
    else:
        transformers.logging.set_verbosity_info()


# Early stopping based on training loss convergence.
# With ~7473 samples, effective batch=16 → ~467 steps/epoch.
# WINDOW=20 covers ~4% of one epoch — enough to confirm a real plateau, not a transient dip.
# MIN_DELTA=0.005: stop if loss hasn't dropped by more than 0.005 over those 20 log events.
EARLY_STOP_WINDOW = 20
EARLY_STOP_MIN_DELTA = 0.005

all_summaries = []

for run_index, overrides in enumerate(SWEEP_RUNS):
    merged = merged_training_params(overrides)
    out = (
        Path(OUTPUT_BASE) / combination_subdir(run_index, merged)
        if RUN_HYPERPARAMETER_SWEEP
        else Path(OUTPUT_BASE)
    )
    out.mkdir(parents=True, exist_ok=True)
    log_file = out / "training_run.log"

    print(
        f"\n=== Run {run_index + 1}/{NUM_COMBINATIONS}  {out.name}  merged={merged} ===\n",
        flush=True,
    )
    _setup_run_logging(log_file)
    logging.info("Logging to %s", log_file.resolve())

    model, tokenizer = download_qwen_25_07b(
        cache_dir=MODEL_CACHE_DIR,
        load_in_4bit=LOAD_IN_4BIT,
        device_map="auto" if LOAD_IN_4BIT else None,
    )
    tokenized_train = load_and_tokenize_math(
        tokenizer,
        name=DATASET_NAME,
        split=DATASET_SPLIT,
        max_samples=merged["max_train_samples"],
        max_length=merged["max_length"],
        message_formatter=format_gsm8k_as_chat,
    )
    cap_msg = (
        "full train split"
        if merged["max_train_samples"] is None
        else f"up to {merged['max_train_samples']} samples"
    )
    logging.info(
        "GSM8K training: %s samples (%s), %s epoch(s).",
        len(tokenized_train),
        cap_msg,
        merged["num_epochs"],
    )

    peft_model = create_lora_model(
        model,
        r=merged["lora_r"],
        lora_alpha=merged["lora_alpha"],
        use_4bit_or_8bit=LOAD_IN_4BIT,
    )
    callbacks = [LossConvergenceCallback(window=EARLY_STOP_WINDOW, min_delta=EARLY_STOP_MIN_DELTA)]
    if SHOW_NOTEBOOK_LOG_LINES:
        callbacks.append(NotebookProgressCallback())

    metrics = run_finetune(
        peft_model,
        tokenizer,
        tokenized_train,
        output_dir=str(out),
        num_epochs=merged["num_epochs"],
        per_device_train_batch_size=merged["per_device_train_batch_size"],
        gradient_accumulation_steps=merged["gradient_accumulation_steps"],
        learning_rate=merged["learning_rate"],
        logging_steps=LOGGING_STEPS,
        logging_first_step=True,
        logging_strategy="steps",
        disable_tqdm=DISABLE_TQDM,
        callbacks=callbacks,
        save_log_history_json=True,
    )

    summary = {
        "run_index": run_index,
        "run_hyperparameter_sweep": RUN_HYPERPARAMETER_SWEEP,
        "sweep_overrides": overrides,
        "merged_params": merged,
        "finished_at_utc": datetime.now(timezone.utc).isoformat(),
        "output_dir": str(out.resolve()),
        "log_files": {
            "training_run_log": str((out / "training_run.log").resolve()),
            "trainer_log_history_json": str((out / "trainer_log_history.json").resolve()),
        },
        "logging_steps": LOGGING_STEPS,
        "show_notebook_log_lines": SHOW_NOTEBOOK_LOG_LINES,
        "disable_tqdm": DISABLE_TQDM,
        "transformers_debug": TRANSFORMERS_DEBUG,
        "early_stop_window": EARLY_STOP_WINDOW,
        "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
        "dataset": DATASET_NAME,
        "split": DATASET_SPLIT,
        "num_train_samples": len(tokenized_train),
        "num_epochs": merged["num_epochs"],
        "per_device_train_batch_size": merged["per_device_train_batch_size"],
        "gradient_accumulation_steps": merged["gradient_accumulation_steps"],
        "learning_rate": merged["learning_rate"],
        "max_length": merged["max_length"],
        "lora_r": merged["lora_r"],
        "lora_alpha": merged["lora_alpha"],
        "load_in_4bit": LOAD_IN_4BIT,
        "final_train_loss": metrics.get("train_loss"),
        "global_step": metrics.get("global_step"),
        "epoch": metrics.get("epoch"),
    }
    with open(out / "training_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    all_summaries.append(summary)

    logging.info("Done run %s/%s: %s", run_index + 1, NUM_COMBINATIONS, out.resolve())
    del model, peft_model, tokenizer, tokenized_train
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if RUN_HYPERPARAMETER_SWEEP:
    sweep_index_path = Path(OUTPUT_BASE) / "sweep_index.json"
    with open(sweep_index_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                "num_runs": NUM_COMBINATIONS,
                "runs": [
                    {
                        "output_dir": s["output_dir"],
                        "final_train_loss": s.get("final_train_loss"),
                        "merged_params": s["merged_params"],
                    }
                    for s in all_summaries
                ],
            },
            f,
            indent=2,
        )
    print(f"Wrote sweep index: {sweep_index_path.resolve()}")

print(
    f"Finished {NUM_COMBINATIONS} run(s). Artifacts under each run folder in:\n  {Path(OUTPUT_BASE).resolve()}\n"
)


=== Run 1/5  run_000_lr1e-5_r8_a16_ep1  merged={'learning_rate': 1e-05, 'lora_r': 8, 'lora_alpha': 16, 'num_epochs': 1, 'per_device_train_batch_size': 4, 'gradient_accumulation_steps': 4, 'max_length': 512, 'max_train_samples': None} ===



2026-04-02 21:53:43 | INFO     | root | Logging to /common/home/users/c/chinyee.lew.2022/output/fine_tune/run_000_lr1e-5_r8_a16_ep1/training_run.log
2026-04-02 21:53:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 21:53:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-04-02 21:53:44 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
loading configuration file config.json from cache at /common/home/users/c/chinyee.lew.2022/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be50417f59c2c2f167def9a775/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2

[train] started | epochs=1 | log every 1 step(s) | progress bar + metrics lines below


Step,Training Loss
1,1.426082
2,1.575900
3,1.548094
4,1.516352
5,1.436841
6,1.435524
7,1.431913
8,1.369368
9,1.489251
10,1.532949


[train] step=1 | loss=1.4261 | lr=0.00e+00 | epoch=0.0021
[train] step=2 | loss=1.5759 | lr=2.17e-07 | epoch=0.0043
[train] step=3 | loss=1.5481 | lr=4.35e-07 | epoch=0.0064
[train] step=4 | loss=1.5164 | lr=6.52e-07 | epoch=0.0086
[train] step=5 | loss=1.4368 | lr=8.70e-07 | epoch=0.0107
[train] step=6 | loss=1.4355 | lr=1.09e-06 | epoch=0.0128
[train] step=7 | loss=1.4319 | lr=1.30e-06 | epoch=0.0150
[train] step=8 | loss=1.3694 | lr=1.52e-06 | epoch=0.0171
[train] step=9 | loss=1.4893 | lr=1.74e-06 | epoch=0.0193
[train] step=10 | loss=1.5329 | lr=1.96e-06 | epoch=0.0214
[train] step=11 | loss=1.5082 | lr=2.17e-06 | epoch=0.0235
[train] step=12 | loss=1.7010 | lr=2.39e-06 | epoch=0.0257
[train] step=13 | loss=1.4807 | lr=2.61e-06 | epoch=0.0278
[train] step=14 | loss=1.5720 | lr=2.83e-06 | epoch=0.0300
[train] step=15 | loss=1.6400 | lr=3.04e-06 | epoch=0.0321
[train] step=16 | loss=1.5696 | lr=3.26e-06 | epoch=0.0342
[train] step=17 | loss=1.5681 | lr=3.48e-06 | epoch=0.0364
[train

...

[train] step=20 | loss=1.5303 | lr=4.07e-06 | epoch=0.0428




Training completed. Do not forget to share your model on huggingface.co/models =)




[train] step=20 | epoch=0.0428
[train] finished | global_step=20


Saving model checkpoint to output/fine_tune/run_004_lr3e-5_r32_a64_ep3
2026-04-02 21:56:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 21:56:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
2026-04-02 21:56:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-02 21:56:36 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B-Instruct/7ae557604adf67be50417f59c2c2f167def9a775/config.json "HTTP/1.1 200 OK"
loading configuration file config.json from cache at /common/home/users/c/chinyee.lew.2022/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct/snapshots/7ae557604adf67be504

Wrote sweep index: /common/home/users/c/chinyee.lew.2022/output/fine_tune/sweep_index.json
Finished 5 run(s). Artifacts under each run folder in:
  /common/home/users/c/chinyee.lew.2022/output/fine_tune

